# FrameSight — Complete Colab Notebook

Use this notebook **in order** (Runtime → Run all, or run each section top to bottom).

| Section | What it does |
|---------|----------------|
| **Before Colab** | Prepare videos + dataset on your Windows PC |
| **Steps 0–2** | GPU + install + training settings |
| **Step 3** | Upload your labeled dataset |
| **Steps 4–5** | Verify dataset + preview labels |
| **Steps 6–8** | Train YOLO11 + view results + test |
| **Step 9** | Download `best.pt` |
| **Step 10** | Start the overlay on Windows |

**Classes:** `body`, `head`

---
## BEFORE COLAB — Run on your Windows PC

Do this once per batch of footage (not inside Colab):

```powershell
cd C:\Users\jonah\Documents\FrameSight

# First time only
.\scripts\setup_windows.ps1
pip install -r requirements-autodistill.txt

# 1) Put .mp4 / .mkv clips in data\videos\
# 2) Extract frames + AutoDistill labels + zip for Colab
python scripts\autodistill_pipeline.py --zip
```

You should get: **`data\autodistill\framesight_dataset.zip`** — upload that in **Step 3** below.

Optional: edit labeling prompts in `config\autodistill.yaml` before running the pipeline.

---
## STEP 0 — Enable GPU (required)

1. Menu: **Runtime → Change runtime type**
2. **Hardware accelerator:** GPU (T4 or better)
3. Click **Save**

Then run the cell below.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU — enable GPU in Runtime settings (Step 0)"
print("GPU ready:", torch.cuda.get_device_name(0))

---
## STEP 1 — Install packages

In [ ]:
!pip install -q ultralytics>=8.3.0 pyyaml opencv-python-headless matplotlib
print("Installed: ultralytics, pyyaml, opencv, matplotlib")

---
## STEP 2 — Training settings

Change these if you run out of GPU memory (lower `BATCH` to 8 or 4).

In [ ]:
# --- edit if needed ---
EPOCHS = 100
BATCH = 16
IMGSZ = 640
PATIENCE = 20
BASE_MODEL = "yolo11n.pt"  # nano = fastest; yolo11s.pt for more accuracy

print(f"Train {BASE_MODEL}  epochs={EPOCHS}  batch={BATCH}  imgsz={IMGSZ}")

---
## STEP 3 — Upload dataset

**Option A (recommended):** Upload `framesight_dataset.zip` from your PC (`python scripts/autodistill_pipeline.py --zip`).

**Option B:** If the zip is already on Google Drive, set `USE_GOOGLE_DRIVE = True` and `DRIVE_ZIP_PATH` in the cell below.

In [ ]:
import zipfile
import shutil
from pathlib import Path

# --- Option B: Google Drive ---
USE_GOOGLE_DRIVE = False
DRIVE_ZIP_PATH = "/content/drive/MyDrive/framesight_dataset.zip"  # change if needed

CONTENT = Path("/content")
DATASET_ZIP = CONTENT / "framesight_dataset.zip"
DATASET_DIR = CONTENT / "dataset"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy(DRIVE_ZIP_PATH, DATASET_ZIP)
    print("Copied from Drive:", DATASET_ZIP)
elif not DATASET_ZIP.exists():
    from google.colab import files
    print("Click 'Choose Files' and select framesight_dataset.zip from your PC")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No file uploaded")
    name = list(uploaded.keys())[0]
    if name != DATASET_ZIP.name:
        shutil.move(name, DATASET_ZIP)
    print("Uploaded:", DATASET_ZIP)
else:
    print("Using existing zip:", DATASET_ZIP)

---
## STEP 4 — Unzip and verify dataset

In [ ]:
import yaml

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
    zf.extractall(CONTENT)

DATASET_DIR = None
for candidate in [
    CONTENT / "framesight_dataset",
    CONTENT / "dataset",
    CONTENT / "data" / "autodistill" / "dataset",
]:
    if (candidate / "data.yaml").exists():
        DATASET_DIR = candidate
        break

if DATASET_DIR is None:
    raise FileNotFoundError(
        "data.yaml not found in zip. Re-run on PC: python scripts/autodistill_pipeline.py --zip"
    )

data_yaml = DATASET_DIR / "data.yaml"
with data_yaml.open() as f:
    cfg = yaml.safe_load(f)
cfg["path"] = str(DATASET_DIR.resolve())
with data_yaml.open("w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

def count_split(name):
    img = DATASET_DIR / name / "images"
    lbl = DATASET_DIR / name / "labels"
    n_img = len(list(img.glob("*"))) if img.is_dir() else 0
    n_lbl = len(list(lbl.glob("*.txt"))) if lbl.is_dir() else 0
    return n_img, n_lbl

tr_i, tr_l = count_split("train")
va_i, va_l = count_split("valid")

print("Dataset folder:", DATASET_DIR)
print("Classes:", cfg.get("names"), "  nc=", cfg.get("nc"))
print(f"train: {tr_i} images, {tr_l} labels")
print(f"valid: {va_i} images, {va_l} labels")

if tr_i == 0:
    raise ValueError("No training images — check your zip / PC pipeline")
if tr_l == 0:
    raise ValueError("No training labels — AutoDistill may have failed on PC")

---
## STEP 5 — Preview labeled frames (optional)

Spot-check that players (`body`) and heads (`head`) look reasonable.

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

CLASS_NAMES = cfg.get("names", ["body", "head"])
COLORS = [(255, 64, 64), (255, 200, 0)]

train_img_dir = DATASET_DIR / "train" / "images"
train_lbl_dir = DATASET_DIR / "train" / "labels"
samples = random.sample(list(train_img_dir.glob("*.jpg")), min(6, len(list(train_img_dir.glob("*.jpg")))))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, img_path in zip(axes, samples):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl_path = train_lbl_dir / f"{img_path.stem}.txt"
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(img_path.name[:28], fontsize=8)
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            cid = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h
            rect = Rectangle((x1, y1), bw * w, bh * h, fill=False,
                             edgecolor=tuple(c/255 for c in COLORS[cid % len(COLORS)]), linewidth=2)
            ax.add_patch(rect)

plt.suptitle("Sample train labels", fontsize=12)
plt.tight_layout()
plt.show()

---
## STEP 6 — Train YOLO11

This may take **1–3+ hours** depending on dataset size and GPU.

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    project=str(CONTENT / "runs"),
    name="framesight",
    exist_ok=True,
)

RUN_DIR = Path(results.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

print("\nTraining done.")
print("  Best:", BEST_PT)
print("  Last:", LAST_PT)
print("  Run folder:", RUN_DIR)

---
## STEP 7 — Training charts

In [ ]:
from IPython.display import Image, display

for name in ["results.png", "confusion_matrix.png", "BoxF1_curve.png", "BoxPR_curve.png"]:
    p = RUN_DIR / name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))

---
## STEP 8 — Test inference on one validation image

In [ ]:
val_dir = DATASET_DIR / "valid" / "images"
if not val_dir.is_dir() or not any(val_dir.iterdir()):
    val_dir = DATASET_DIR / "train" / "images"

test_img = sorted(val_dir.glob("*.jpg"))[0]
trained = YOLO(str(BEST_PT))
pred = trained.predict(str(test_img), imgsz=IMGSZ, conf=0.35, verbose=False)

annotated = pred[0].plot()
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"Predictions: {test_img.name}")
plt.show()

---
## STEP 9 — Download `best.pt` to your PC

Save the file into your FrameSight project:

`C:\Users\jonah\Documents\FrameSight\weights\best.pt`

In [ ]:
from google.colab import files

assert BEST_PT.exists(), f"Missing {BEST_PT} — run Step 6 first"
print("Downloading best.pt ...")
files.download(str(BEST_PT))
print("Done. Place the file in:  FrameSight/weights/best.pt")

---
## STEP 10 — Start the overlay on Windows

After `best.pt` is in `weights\`, run in **PowerShell**:

```powershell
cd C:\Users\jonah\Documents\FrameSight

# First time only
.\scripts\setup_windows.ps1
pip install -r requirements.txt

# Every time — start overlay
.\scripts\run.ps1
```

Or manually:

```powershell
cd C:\Users\jonah\Documents\FrameSight
.\.venv\Scripts\Activate.ps1
$env:ULTRALYTICS_SKIP_REQUIREMENTS_CHECKS = "1"
python -m src.main
```

Stop with **Ctrl+C**.

### Config (optional)

```powershell
copy config\local.yaml.example config\local.yaml
```

Edit `config\default.yaml` or `config\local.yaml` for aim assist, overlay lines, model confidence, etc.

### Quick checks

```powershell
Test-Path weights\best.pt
.\.venv\Scripts\python.exe -c "import ultralytics, dxcam; print('OK')"
```

---
## Quick reference

| Step | Action |
|------|--------|
| PC | Videos → `data/videos/` → `python scripts/autodistill_pipeline.py --zip` |
| Colab 0 | Runtime → GPU |
| Colab 1–2 | Install + settings |
| Colab 3–4 | Upload zip + verify |
| Colab 6 | Train |
| Colab 9 | Download `best.pt` |
| PC | `weights/best.pt` → `.\scripts\run.ps1` |